# **DATA MODELING AND MODEL EVALUATION**

---

## **1. Import Libraries**

In [ ]:
# Standard data manipulation and visualization libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Scikit-learn utilities for preprocessing and evaluation
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import StratifiedKFold

# Import custom models implemented from scratch
from utils import Perceptron, LogisticRegression 

# Configure visualization settings
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_context("notebook", font_scale=1.1)

## **2. Load Data for Model**

Dữ liệu đã được chia sẵn thành 3 tập: Huấn luyện (Train), Thẩm định (Validation), và Kiểm tra (Test).

**Lưu ý quan trọng về Tiền xử lý:** Các thuật toán tối ưu hóa dựa trên Gradient (như Perceptron và Logistic Regression) cực kỳ nhạy cảm với thang đo của dữ liệu. Nếu không chuẩn hóa, các đặc trưng có giá trị lớn (như CO2) sẽ chi phối hoàn toàn quá trình cập nhật trọng số, làm cho hàm mất mát mất đi tính đối xứng và thuật toán rất khó hội tụ. Do đó, kỹ thuật `StandardScaler` (đưa kỳ vọng về 0, phương sai về 1) được áp dụng. Để tránh rò rỉ dữ liệu (Data Leakage), bộ chuẩn hóa chỉ được `fit` trên tập Train, sau đó dùng để `transform` cho cả 3 tập.

In [ ]:
# Define file paths
train_path = '../../data/processed/Room_Occupancy_train.csv'
val_path = '../../data/processed/Room_Occupancy_val.csv'
test_path = '../../data/processed/Room_Occupancy_test.csv'

# Load the datasets
train_df = pd.read_csv(train_path)
val_df = pd.read_csv(val_path)
test_df = pd.read_csv(test_path)

# Define the target variable
target_col = 'Room_Occupancy_Count'

# Split features and target labels
X_train = train_df.drop(columns=[target_col]).values
y_train = train_df[target_col].values

X_val = val_df.drop(columns=[target_col]).values
y_val = val_df[target_col].values

X_test = test_df.drop(columns=[target_col]).values
y_test = test_df[target_col].values

# Standardization to handle varying scales of sensor data (e.g., Light vs CO2)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

# Display the shape of the datasets
print(f"Training set shape:   {X_train_scaled.shape}")
print(f"Validation set shape: {X_val_scaled.shape}")
print(f"Testing set shape:    {X_test_scaled.shape}")

## **3. Thuật toán Perceptron: Quan sát sự hội tụ**

Thuật toán Perceptron nguyên bản (Rosenblatt, 1958) hoạt động dựa trên cơ chế sửa lỗi trực tiếp: nó chỉ cập nhật vector trọng số khi phát hiện một điểm dữ liệu bị phân lớp sai. 

Theo **Định lý hội tụ Perceptron**, nếu tập dữ liệu hoàn toàn phân tách tuyến tính (linearly separable), thuật toán chắc chắn sẽ hội tụ và tìm ra một siêu phẳng phân chia hoàn hảo sau một số bước lặp hữu hạn (số lỗi giảm về 0). Tuy nhiên, nếu dữ liệu có nhiễu hoặc các vùng giao thoa, thuật toán sẽ dao động không ngừng. Chúng ta sẽ tiến hành huấn luyện và vẽ đồ thị số lượng mẫu phân loại sai qua từng Epoch để kiểm chứng tính chất hình học của bộ dữ liệu `Room_Occupancy`.

In [ ]:
# Initialize and train the Custom Perceptron
# Note: Ensure your Perceptron implementation in utils.py saves the error count per epoch
learning_rate = 0.01
max_epochs = 1000

print("Training Custom Perceptron...")
perceptron_model = Perceptron(learning_rate=learning_rate, max_iter=max_epochs)
perceptron_model.fit(X_train_scaled, y_train)

# Retrieve the history of misclassifications from the model (adjust attribute name if needed)
# Assuming it's stored in perceptron_model.errors_history
errors = perceptron_model.errors_history 
epochs_run = len(errors)

print(f"Training stopped after {epochs_run} epochs.")
print(f"Final misclassified samples on training set: {errors[-1]}")

# Evaluate basic accuracy on the test set
y_pred_test_perc = perceptron_model.predict(X_test_scaled)
test_acc_perc = accuracy_score(y_test, y_pred_test_perc)
print(f"Perceptron Test Accuracy: {test_acc_perc * 100:.2f}%\n")

# Plot the Convergence Curve
plt.figure(figsize=(10, 6))
plt.plot(range(1, epochs_run + 1), errors, color='crimson', linewidth=1.5, alpha=0.8)

plt.title('Perceptron Convergence Curve (Misclassifications per Epoch)', fontsize=14, fontweight='bold')
plt.xlabel('Epochs', fontsize=12)
plt.ylabel('Number of Errors', fontsize=12)
plt.axhline(y=0, color='black', linestyle='--', linewidth=1, label='Zero Error Line')

# Add annotation for the final state
plt.annotate(f'Final: {errors[-1]} errors', 
             xy=(epochs_run, errors[-1]), 
             xytext=(epochs_run * 0.8, errors[-1] + (max(errors) * 0.1)),
             arrowprops=dict(facecolor='black', shrink=0.05, width=1.5, headwidth=8),
             fontsize=12, fontweight='bold', color='darkred')

plt.legend()
plt.tight_layout()
plt.show()

**Phân tích sự hội tụ và đánh giá hiệu năng của Perceptron:**

1. **Sự hội tụ hoàn hảo trên tập Huấn luyện (Train):** Đồ thị cho thấy một sự sụt giảm cực kỳ dốc và chạm mốc `0` lỗi chỉ sau vỏn vẹn 17 vòng lặp (Epochs). Điều này chứng minh bằng thực nghiệm **Định lý hội tụ Novikoff**: Tập dữ liệu huấn luyện `Room_Occupancy` hoàn toàn **có thể phân tách tuyến tính (linearly separable)**. Perceptron đã thành công trong việc tìm ra một siêu phẳng chia cắt 100% dữ liệu Train mà không có điểm lấn tuyến.

2. **Hiện tượng Quá khớp (Overfitting) nghiêm trọng:** Dù đạt 0 lỗi trên tập Train, độ chính xác trên tập Kiểm tra (Test) chỉ đạt **55.33%**. Đây là một sự sụp đổ về mặt hiệu năng khái quát hóa (generalization).

3. **Bản chất toán học của sự thất bại:** Điểm yếu chí mạng của thuật toán Perceptron nguyên bản là nó sẽ **dừng lại ngay lập tức** khi tìm thấy *bất kỳ* ranh giới nào phân tách được dữ liệu. Nó hoàn toàn không có khái niệm tối ưu hóa "lề ranh giới" (margin) hay kiểm soát độ phức tạp. Hệ quả là, siêu phẳng mà nó tìm được ở vòng lặp thứ 17 có thể nằm sát sàn sạt vào một điểm dữ liệu của lớp "Phòng trống", khiến nó phân loại sai thảm hại khi gặp dữ liệu Test hơi lệch đi một chút.

$\Rightarrow$ **Kết luận:** Perceptron quá đơn giản và "bảo thủ". Để mô hình có thể tổng quát hóa tốt hơn trên dữ liệu thực tế và chống lại nhiễu, chúng ta cần một thuật toán đánh giá dựa trên xác suất, có cơ chế xử lý mất cân bằng lớp và tích hợp các hàm phạt (Regularization) như **Logistic Regression**.


## **4. Hồi quy Logistic: Regularization (L1/L2), Class-Weighted Loss và Stratified K-Fold CV**

Trong phần này, chúng ta sử dụng lớp `LogisticRegression` tự cài đặt từ đầu (from scratch) với các tính năng nâng cao:
* **Hàm mất mát có trọng số (Class-Weighted Loss):** Giúp mô hình không bị thiên lệch (bias) khi tập dữ liệu bị mất cân bằng (imbalanced) giữa lớp 0 và lớp 1.
* **Điều chuẩn L1 (Lasso) / L2 (Ridge):** Giới hạn độ lớn của vector trọng số $\mathbf{w}$, ngăn chặn mô hình học vẹt các đặc trưng nhiễu.

Để tìm ra cấu hình tốt nhất (loại Penalty và hệ số $\lambda$), kỹ thuật **Stratified K-Fold Cross Validation (k=5)** được áp dụng. Việc dùng bản `Stratified` sẽ đảm bảo tỷ lệ giữa lớp "Có người" và "Không người" được giữ nguyên vẹn trong từng Fold chia nhỏ, giúp quá trình đánh giá siêu tham số khách quan nhất.

In [ ]:
from sklearn.model_selection import StratifiedKFold
import pandas as pd

# 1. Define the hyperparameter grid
# We will test both L1 and L2 penalties across different lambda (regularization strength) values
penalties = ['l1', 'l2']
lambda_values = [0.001, 0.01, 0.1, 1.0, 10.0]
k_splits = 5

# 2. Initialize Stratified K-Fold
# random_state is set for reproducibility of the folds
skf = StratifiedKFold(n_splits=k_splits, shuffle=True, random_state=42)

# Variables to keep track of the best performing hyperparameter combination
best_cv_acc = 0
best_params = {'penalty': None, 'lambda_reg': None}
cv_results_log = []

print(f"--- Starting Stratified {k_splits}-Fold CV for Logistic Regression ---")
print(f"Features: Standardized | Loss: Class-Weighted (Balanced)")
print("-" * 75)
print(f"{'Penalty':<10} | {'Lambda':<10} | {'Mean CV Accuracy (%)':<25} | {'Std Dev (%)':<15}")
print("-" * 75)

# 3. Grid Search loop
for penalty in penalties:
    for lam in lambda_values:
        fold_accuracies = []
        
        # Split the standardized training data into CV Train and CV Validation folds
        for fold, (train_idx, val_idx) in enumerate(skf.split(X_train_scaled, y_train)):
            X_cv_train, X_cv_val = X_train_scaled[train_idx], X_train_scaled[val_idx]
            y_cv_train, y_cv_val = y_train[train_idx], y_train[val_idx]
            
            # Initialize the custom Logistic Regression model
            # Note: class_weight='balanced' activates the weighted Cross-Entropy loss
            lr_cv_model = LogisticRegression(learning_rate=0.01, 
                                             max_iter=500, # Reduced max_iter for faster CV
                                             penalty=penalty, 
                                             lambda_reg=lam, 
                                             class_weight='balanced')
            
            # Train on the CV Training fold
            lr_cv_model.fit(X_cv_train, y_cv_train)
            
            # Evaluate on the CV Validation fold
            y_pred_val = lr_cv_model.predict(X_cv_val)
            acc = accuracy_score(y_cv_val, y_pred_val)
            fold_accuracies.append(acc)
            
        # Calculate mean and standard deviation of accuracy across all 5 folds
        mean_acc = np.mean(fold_accuracies)
        std_acc = np.std(fold_accuracies)
        
        cv_results_log.append({
            'Penalty': penalty.upper(), 
            'Lambda': lam, 
            'Mean_CV_Accuracy': mean_acc,
            'Std_Dev': std_acc
        })
        
        print(f"{penalty.upper():<10} | {lam:<10} | {mean_acc*100:<25.2f} | {std_acc*100:<15.2f}")
        
        # Update best parameters if current combination is better
        if mean_acc > best_cv_acc:
            best_cv_acc = mean_acc
            best_params = {'penalty': penalty, 'lambda_reg': lam}

print("-" * 75)
print(f">>> BEST PARAMS: Penalty = {best_params['penalty'].upper()}, Lambda = {best_params['lambda_reg']}")
print(f">>> BEST CV ACCURACY: {best_cv_acc*100:.2f}%")

# Convert results to a pandas DataFrame for better visualization later
df_cv_results = pd.DataFrame(cv_results_log)
df_cv_results.sort_values(by='Mean_CV_Accuracy', ascending=False, inplace=True)
df_cv_results.reset_index(drop=True, inplace=True)

**Phân tích kết quả dò tìm siêu tham số (Grid Search CV):**

Dựa vào bảng kết quả Stratified 5-Fold CV, chúng ta rút ra được các quan sát quan trọng về hành vi của mô hình Hồi quy Logistic khi áp dụng hàm phạt (Regularization):

1. **Sự ổn định ở mức phạt thấp:** Cả L1 và L2 đều đạt hiệu năng tối ưu (quanh mức 76.9%) khi hệ số phạt $\lambda$ nhỏ (từ 0.001 đến 0.1). Điều này cho thấy mô hình đã tìm được sự cân bằng tốt giữa việc khớp dữ liệu (fitting) và kiểm soát độ phức tạp.
2. **Sự sụp đổ ở mức phạt cực đại (Underfitting):** Khi đẩy $\lambda$ lên mức `10.0`, độ chính xác của cả L1 và L2 đều sụp đổ thê thảm xuống còn `6.47%`. Về mặt toán học, đây là hiện tượng Underfitting do mức thuế phạt quá nặng đã "đè bẹp" toàn bộ vector trọng số $\mathbf{w}$ về $0$. Mô hình bị mất hoàn toàn khả năng nhận diện đặc trưng và dự đoán mù quáng.
3. **Chiến thắng của Điều chuẩn L1 (Lasso):** Cấu hình tốt nhất thuộc về **Penalty = L1** và **Lambda = 0.001**. Việc L1 chiếm ưu thế ngầm chỉ ra rằng trong tập dữ liệu cảm biến này, tồn tại một số đặc trưng nhiễu hoặc dư thừa. Đặc tính hình học của L1 đã tự động thực hiện việc "chọn lọc đặc trưng" (Feature Selection) bằng cách ép trọng số của các đặc trưng vô dụng về chính xác `0`, giúp mô hình tổng quát hóa tốt hơn L2.

## **5. Đánh giá Mô hình Hồi quy Logistic Tối ưu**

Sau khi xác định được bộ siêu tham số quán quân (`Penalty = L1`, `Lambda = 0.001`), chúng ta tiến hành:
1. Huấn luyện lại mô hình trên toàn bộ tập dữ liệu Huấn luyện (Train).
2. Quan sát đường cong học tập (Learning Curve) qua hàm mất mát (Cross-Entropy Loss) để thấy sự hội tụ mượt mà so với sự dao động của Perceptron.
3. Đánh giá hiệu năng thực tế trên tập Kiểm tra (Test) bằng Ma trận nhầm lẫn (Confusion Matrix) và Báo cáo phân lớp (Classification Report). Quá trình này sẽ cho thấy sức mạnh của việc tích hợp `class_weight='balanced'` trong việc nhận diện lớp thiểu số.

In [ ]:
# Train the best Logistic Regression model
print("Training the best Logistic Regression model...")
best_lr_model = LogisticRegression(
    learning_rate=0.1,  # Increased learning rate for better convergence
    max_iter=1000, 
    penalty='l1', 
    lambda_reg=0.001, 
    class_weight='balanced'
)

best_lr_model.fit(X_train_scaled, y_train)
print("Training completed successfully.\n")

# Evaluate on the test set
y_pred_test_lr = best_lr_model.predict(X_test_scaled)
test_acc_lr = accuracy_score(y_test, y_pred_test_lr)

print("-" * 55)
print(f"Logistic Regression (L1, lam=0.001) Test Accuracy: {test_acc_lr*100:.2f}%")
print("-" * 55)
print("Classification Report:")
# Added labels=[0, 1] to prevent ValueError if model only predicts one class
print(classification_report(y_test, y_pred_test_lr, labels=[0, 1], target_names=['Empty (0)', 'Occupied (1)'], zero_division=0))

# Visualizations: Loss Curve & Confusion Matrix
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

axes[0].plot(range(1, len(best_lr_model.loss_history) + 1), best_lr_model.loss_history, color='teal', linewidth=2)
axes[0].set_title('Logistic Regression: Loss Curve (Convergence)', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Epochs', fontsize=12)
axes[0].set_ylabel('Weighted Cross-Entropy Loss', fontsize=12)

cm = confusion_matrix(y_test, y_pred_test_lr, labels=[0, 1])
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[1], 
            xticklabels=['Empty (0)', 'Occupied (1)'], 
            yticklabels=['Empty (0)', 'Occupied (1)'],
            annot_kws={"size": 14, "weight": "bold"})
axes[1].set_title('Confusion Matrix on Test Set', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Predicted Label', fontsize=12)
axes[1].set_ylabel('True Label', fontsize=12)

plt.tight_layout()
plt.show()

**Phân tích Hiệu năng và Đánh giá Hồi quy Logistic (L1, $\lambda=0.001$):**

1. **Sự hội tụ mượt mà (Loss Curve):** Khác hoàn toàn với sự dao động giật cục của Perceptron, đồ thị Loss Curve của Logistic Regression giảm một cách cực kỳ mượt mà và tiệm cận dần về mức ổn định (convergence). Điều này chứng minh thuật toán Gradient Descent kết hợp với hàm phạt L1 đã hoạt động hoàn hảo trên phương diện toán học, từ từ tối ưu hóa trọng số mà không bị nhiễu đánh lừa.

2. **Ảo ảnh 100% Accuracy và vấn đề phân bố dữ liệu Test:**
   Mô hình đạt độ chính xác (Test Accuracy) lên tới `100.00%`, tuy nhiên, nhìn vào Ma trận nhầm lẫn (Confusion Matrix) và Báo cáo phân lớp (Classification Report), chúng ta phát hiện ra một đặc điểm chí mạng của tập dữ liệu `Room_Occupancy_test.csv`:
   * Toàn bộ 1520 mẫu trong tập Test đều thuộc lớp `Empty (0)`. Lớp `Occupied (1)` có số lượng (support) bằng chính xác `0`.
   * Vì tập Test bị **mất cân bằng tuyệt đối (100% lớp 0)**, mô hình chỉ cần dự đoán tất cả là "Phòng trống" thì tự động sẽ đạt điểm tuyệt đối.

3. **Tổng kết phần Phân lớp:**
   Thuật toán Logistic Regression tự cài đặt từ đầu đã vận hành đúng đắn các cơ chế cốt lõi (Class-weighted loss, Regularization). Tuy nhiên, thực nghiệm cuối cùng này nhấn mạnh một bài học sâu sắc trong Khoa học Dữ liệu: Các chỉ số như Accuracy sẽ hoàn toàn vô nghĩa nếu tập Test không phản ánh đúng phân phối thực tế của dữ liệu. Để đánh giá thực sự sức mạnh của mô hình này (đặc biệt là khả năng nhận diện phòng có người), tập Test cần phải được thu thập lại hoặc trích xuất (split) lại từ đầu để đảm bảo có sự xuất hiện của cả hai lớp.